# ToLD-Br — *Features* Léxicas e Validação População vs. Amostra (Fix B)

Este *notebook* produz a tabela-cabeça da §5 do artigo metodológico: **estatísticas das *features* léxicas sob a população vs. amostra Cochran-por-estrato (Fix B)**, com erros-padrão segundo Cochran (1977) §2.6.

Pipeline:
1. Carregar o ToLD-Br e adicionar cinco *features* léxicas (`word_count`, `unique_word_ratio`, `caps_ratio`, `punc_density`, `mean_word_length`).
2. Construir estratos mutuamente exclusivos pela regra de prioridade *raro vence*: tweets com qualquer rótulo majoritário positivo (votos ≥ 2) são alocados ao rótulo mais raro entre os positivos; senão, ao estrato `clean`.
3. Calcular a alocação Cochran-por-estrato (com censo dos estratos pequenos) e sortear a amostra.
4. Construir a tabela de comparação população vs. amostra por estrato.
5. Tabelar o diagnóstico de ruído por contagem de votos (Camada 3.1) — quantos tweets ficam em zona disputada (exatamente 1 anotador).

In [1]:
from pathlib import Path

import pandas as pd

from toxicity_analysis.constants import TOLDBR_LABELS
from toxicity_analysis.features import (
    add_lexical_features,
    feature_summary,
    vote_noise_breakdown,
)
from toxicity_analysis.sampling import (
    calc_cochran,
    draw_stratified,
    per_stratum_cochran_allocation,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = REPO_ROOT / "data" / "raw" / "told-br" / "train.csv"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = list(TOLDBR_LABELS)
FEATURES = ["word_count", "unique_word_ratio", "caps_ratio", "punc_density", "mean_word_length"]

## 1. Carregar e engenheirar *features*

In [2]:
df = pd.read_csv(RAW_PATH)
df = add_lexical_features(df)
df.head(2)

,text,homophobia,obscene,insult,racism,misogyny,xenophobia,word_count,unique_word_ratio,caps_ratio,punc_density,mean_word_length
0,Meu nivel de amizade com isis é ela ter meu in...,0,0,2,0,0,0,32,0.8125,0.008547,0.006579,3.65625
1,"rt @user @user o cara adultera dados, que fora...",0,0,1,0,0,0,16,0.9375,0.000000,0.043478,4.56250


## 2. Estratos mutuamente exclusivos pela regra *raro vence*

Atribuímos cada tweet a um único estrato para que a amostragem particione a população. A ordem de prioridade vai do mais raro ao mais comum: `racism → xenophobia → misogyny → homophobia → insult → obscene → clean`. Isso garante que os estratos raros mantenham toda a sua contagem populacional (sustenta a afirmação do artigo de capturar a cauda).

In [3]:
PRIORIDADE = ["racism", "xenophobia", "misogyny", "homophobia", "insult", "obscene"]


def estrato_de(row: pd.Series) -> str:
    for label in PRIORIDADE:
        if row[label] >= 2:
            return label
    return "clean"


df["estrato"] = df.apply(estrato_de, axis=1)
df["estrato"].value_counts().reindex([*PRIORIDADE, "clean"])

estrato
racism           33
xenophobia       42
misogyny        129
homophobia      172
insult         1755
obscene        1932
clean         16937
Name: count, dtype: int64

## 3. Tamanho amostral de Cochran e alocação Fix B

O tamanho amostral global por Cochran (95% IC, 5% de margem, p=0,5) fornece o número de cabeçalho `n=378`. Mas ele *não* é usado para alocação proporcional — em vez disso, aplicamos Cochran por estrato e fazemos censo dos pequenos (Fix B).

In [4]:
headline = calc_cochran(N=len(df))
print(f"Cochran populacional: n0={headline.n0:.1f}, n={headline.n} (z={headline.z:.3f})")

alloc = per_stratum_cochran_allocation(df, "estrato")
alloc_df = (
    pd.DataFrame({
        "N_h": df["estrato"].value_counts(),
        "n_h": pd.Series(alloc),
    })
    .reindex([*PRIORIDADE, "clean"])
)
alloc_df["regra"] = alloc_df.apply(lambda r: "censo" if r["n_h"] == r["N_h"] else "amostra", axis=1)
alloc_df.loc["TOTAL"] = [alloc_df["N_h"].sum(), alloc_df["n_h"].sum(), ""]
alloc_df

Cochran populacional: n0=384.1, n=378 (z=1.960)


,N_h,n_h,regra
racism,33,31,amostra
xenophobia,42,38,amostra
misogyny,129,97,amostra
homophobia,172,120,amostra
insult,1755,316,amostra
obscene,1932,321,amostra
clean,16937,376,amostra
TOTAL,21000,1299,


## 4. Sortear a amostra Fix B

In [5]:
amostra_b = draw_stratified(df, "estrato", alloc, random_state=42)
print(f"Amostra Fix B: n={len(amostra_b)}")
amostra_b["estrato"].value_counts().reindex([*PRIORIDADE, "clean"])

Amostra Fix B: n=1299


estrato
racism         31
xenophobia     38
misogyny       97
homophobia    120
insult        316
obscene       321
clean         376
Name: count, dtype: int64

## 5. População vs. amostra Fix B — médias por estrato

Validação: uma amostra estratificada é *representativa* se as suas médias por estrato caem dentro do erro-padrão da população. Esta é a tabela que vai para a §5 do artigo.

In [6]:
pop = feature_summary(df, FEATURES, group_col="estrato").round(3)
smp = feature_summary(amostra_b, FEATURES, group_col="estrato").round(3)

comparacao = (
    pop.add_suffix("_pop")
    .join(smp.add_suffix("_amo"))
    .reindex([*PRIORIDADE, "clean"])
)
comparacao.to_csv(OUT_DIR / "feature_comparison_pop_vs_sampleB.csv")
comparacao

,word_count_mean_pop,unique_word_ratio_mean_pop,caps_ratio_mean_pop,punc_density_mean_pop,mean_word_length_mean_pop,word_count_sem_pop,unique_word_ratio_sem_pop,caps_ratio_sem_pop,punc_density_sem_pop,mean_word_length_sem_pop,...,unique_word_ratio_mean_amo,caps_ratio_mean_amo,punc_density_mean_amo,mean_word_length_mean_amo,word_count_sem_amo,unique_word_ratio_sem_amo,caps_ratio_sem_amo,punc_density_sem_amo,mean_word_length_sem_amo,n_amo
estrato,,,,,,,,,,,,,,,,,,,,,
racism,19.303,0.904,0.013,0.039,4.282,2.792,0.018,0.010,0.011,0.108,...,0.903,0.014,0.040,4.297,2.745,0.019,0.011,0.011,0.114,31
xenophobia,15.810,0.941,0.003,0.037,4.613,1.639,0.013,0.002,0.006,0.140,...,0.937,0.003,0.037,4.643,1.793,0.014,0.002,0.006,0.149,38
misogyny,14.419,0.948,0.010,0.036,4.540,1.100,0.007,0.005,0.003,0.091,...,0.948,0.013,0.033,4.618,1.347,0.008,0.007,0.004,0.110,97
homophobia,15.006,0.945,0.011,0.045,4.242,0.866,0.006,0.004,0.004,0.052,...,0.952,0.005,0.044,4.211,0.963,0.007,0.002,0.004,0.054,120
insult,17.104,0.935,0.022,0.041,4.318,0.309,0.002,0.002,0.001,0.022,...,0.934,0.034,0.041,4.370,0.742,0.005,0.008,0.002,0.063,316
obscene,15.293,0.945,0.017,0.038,4.134,0.243,0.002,0.002,0.001,0.031,...,0.942,0.010,0.038,4.032,0.587,0.005,0.003,0.002,0.042,321
clean,16.039,0.946,0.014,0.039,4.321,0.092,0.001,0.001,0.000,0.008,...,0.950,0.015,0.037,4.289,0.637,0.004,0.003,0.002,0.048,376


In [7]:
# Verificação: médias amostrais ficam dentro de 1 erro-padrão populacional?
ok = pd.DataFrame(index=comparacao.index)
for f in FEATURES:
    diff = (comparacao[f"{f}_mean_pop"] - comparacao[f"{f}_mean_amo"]).abs()
    ok[f] = diff <= comparacao[f"{f}_sem_pop"]
ok

,word_count,unique_word_ratio,caps_ratio,punc_density,mean_word_length
estrato,,,,,
racism,True,True,True,True,True
xenophobia,True,True,True,True,True
misogyny,True,True,True,True,True
homophobia,True,False,False,True,True
insult,False,True,False,True,False
obscene,False,False,False,True,False
clean,False,False,True,False,False


## 6. Diagnóstico de ruído por contagem de votos (Camada 3.1)

Para cada categoria, a fração de tweets com exatamente um voto de anotador — a zona *disputada* que a regra de maioria descarta como negativa, mas que um limiar mais estrito ou mais frouxo inverteria. Diagnóstico de ruído de rotulagem específico ao esquema de três anotadores do ToLD-Br.

In [8]:
ruido = vote_noise_breakdown(df).round(4)
ruido.to_csv(OUT_DIR / "vote_noise_breakdown.csv")
ruido

,votes_0,votes_1,votes_2,votes_3,positives_majority,disputed_share
category,,,,,,
homophobia,20656,168,102,74,176,0.0080
obscene,14348,4249,1791,612,2403,0.2023
insult,16615,2516,1352,517,1869,0.1198
racism,20862,105,27,6,33,0.0050
misogyny,20537,330,104,29,133,0.0157
xenophobia,20849,109,27,15,42,0.0052


## Saídas

- `data/processed/feature_comparison_pop_vs_sampleB.csv` — importação direta para a tabela de validação da §5 do artigo.
- `data/processed/vote_noise_breakdown.csv` — alimenta a tabela de diagnóstico de ruído da §5.4.

## Próximas análises (notebook `03`)

- Alocação ótima de Neyman.
- Estimadores tipo razão e tipo regressão sob desenho estratificado, conforme Cochran (1977) §6 e §7.
- Comparação das estimativas (Neyman vs. proporcional × razão vs. regressão × média estratificada simples).